# Phase 2 - Data cleaning with PySpark

**Input:** the 1% Electronics sample from Phase 1 (`MyDrive/amazon_reviews/sample/`)
**Output:** two clean Parquet tables that every module will use

| Table | One row per | Used by |
|---|---|---|
| `reviews_clean` | review | EDA, sentiment, fake reviews, recommendation |
| `products_clean` | product | product names, category and price analysis |

Steps: set up Spark -> load JSON -> remove bad rows -> clean text -> tokens and stop words -> sentiment label -> save Parquet -> check.

Run the cells **top to bottom**. The whole notebook takes about 10-15 minutes on the 1% sample.

## 1. Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/amazon_reviews'
SAMPLE = '1pct'                      # change to '5pct' later for a bigger sample
CATEGORY = 'Electronics'

RAW_LOCAL = '/content/data'          # fast local copy of the input
OUT_LOCAL = '/content/parquet'       # Spark writes here first, then we copy to Drive

## 2. Install Java 17 and PySpark

Spark runs on Java. Spark 3.5 supports Java 8, 11 and 17, so we install Java 17 to be safe.
Takes about 1-2 minutes.

In [ ]:
import os
!apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip -q install pyspark==3.5.3
!java -version

## 3. Copy the sample from Drive to the local disk (much faster for Spark)

In [ ]:
!mkdir -p {RAW_LOCAL}
!cp {PROJECT}/sample/{CATEGORY}_{SAMPLE}.jsonl.gz {PROJECT}/sample/meta_{CATEGORY}_{SAMPLE}.jsonl.gz {RAW_LOCAL}/
!ls -lh {RAW_LOCAL}

## 4. Start Spark

- `local[*]` = use all CPU cores of this Colab machine
- `spark.driver.memory = 8g` = Colab has about 12 GB RAM, we give Spark most of it
- `shuffle.partitions = 16` = the default (200) is too many for a small machine

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .appName('Phase2-Cleaning')
         .master('local[*]')
         .config('spark.driver.memory', '8g')
         .config('spark.sql.shuffle.partitions', '16')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

## 5. Load the raw JSON with a fixed schema

Giving Spark the schema (column names and types) is faster than letting it guess, and it
skips columns we do not need (for example `images`).

`price` is read as text because some products have no price; we convert it to a number later.

In [ ]:
review_schema = T.StructType([
    T.StructField('rating', T.DoubleType()),
    T.StructField('title', T.StringType()),
    T.StructField('text', T.StringType()),
    T.StructField('asin', T.StringType()),
    T.StructField('parent_asin', T.StringType()),
    T.StructField('user_id', T.StringType()),
    T.StructField('timestamp', T.LongType()),
    T.StructField('helpful_vote', T.IntegerType()),
    T.StructField('verified_purchase', T.BooleanType()),
])

meta_schema = T.StructType([
    T.StructField('parent_asin', T.StringType()),
    T.StructField('title', T.StringType()),
    T.StructField('main_category', T.StringType()),
    T.StructField('average_rating', T.DoubleType()),
    T.StructField('rating_number', T.LongType()),
    T.StructField('price', T.StringType()),
    T.StructField('store', T.StringType()),
    T.StructField('categories', T.ArrayType(T.StringType())),
])

reviews_raw = spark.read.schema(review_schema).json(f'{RAW_LOCAL}/{CATEGORY}_{SAMPLE}.jsonl.gz').cache()
meta_raw = spark.read.schema(meta_schema).json(f'{RAW_LOCAL}/meta_{CATEGORY}_{SAMPLE}.jsonl.gz').cache()

reviews_raw.printSchema()
reviews_raw.show(3, truncate=60)

## 6. Remove bad rows (and record how many rows each step removes)

The cleaning log goes into the report as a table: *rows before and after each step*.

**Duplicates:** we only remove a review if the **same user** wrote the **same text** on the
**same product** (a double submission). The same text on *different* products is **kept**,
because that is one of the fake-review signals for Phase 4.

In [ ]:
cleaning_log = []

def log_step(name, df):
    n = df.count()
    cleaning_log.append((name, n))
    print(f'{name:<50}{n:>12,}')
    return df

df = log_step('1. Raw reviews', reviews_raw)

df = log_step('2. Drop rows missing user, product or rating',
              df.dropna(subset=['user_id', 'parent_asin', 'rating']))

df = log_step('3. Keep ratings between 1 and 5',
              df.filter(F.col('rating').between(1, 5)))

df = log_step('4. Remove exact duplicates (same user+product+text)',
              df.dropDuplicates(['user_id', 'parent_asin', 'text']))

## 7. Clean the review text

What we do, in order:
1. Join the review **title** and **text** (the title often has the strongest opinion, e.g. "Waste of money")
2. Remove HTML tags like `<br />` and HTML codes like `&#34;`
3. Remove links
4. Lowercase
5. Turn **n't** into **not** (`don't` -> `do not`) so negation is not lost
6. Keep only letters and spaces, then squeeze extra spaces

We also add: a stable `review_id`, the review date and year, and the word count.

In [ ]:
def clean_text(title_col, text_col):
    c = F.concat_ws(' ', F.coalesce(title_col, F.lit('')), F.coalesce(text_col, F.lit('')))
    c = F.regexp_replace(c, r'<[^>]+>', ' ')                 # HTML tags such as <br />
    c = F.regexp_replace(c, r'&#?[a-zA-Z0-9]+;', ' ')        # HTML codes such as &#34;
    c = F.regexp_replace(c, r'https?://\S+|www\.\S+', ' ')  # links
    c = F.lower(c)
    c = F.regexp_replace(c, r"n't\b", ' not')               # don't -> do not
    c = F.regexp_replace(c, r'[^a-z ]', ' ')                 # keep letters only
    c = F.regexp_replace(c, r'\s+', ' ')
    return F.trim(c)

df = (df
      .withColumn('review_id', F.substring(F.sha2(F.concat_ws('|', 'user_id', 'parent_asin', F.col('timestamp').cast('string')), 256), 1, 16))
      .withColumn('review_date', F.to_date(F.from_unixtime(F.col('timestamp') / 1000)))
      .withColumn('year', F.year('review_date'))
      .withColumn('text_clean', clean_text(F.col('title'), F.col('text'))))

df = log_step('5. Drop reviews with no words after cleaning',
              df.filter(F.length('text_clean') > 0))

df.select('title', 'text', 'text_clean').show(5, truncate=70)

## 8. Split into words and remove stop words

- **RegexTokenizer** splits the text into words (and drops 1-letter words)
- **StopWordsRemover** removes very common words (*the, is, and, ...*) that carry no opinion

**Important:** Spark's default stop-word list contains *not, no, very, too*. Those words change
the meaning ("not good" vs "good"), so we **keep** them. This is a good point to explain in the viva.

In [ ]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

KEEP_WORDS = {'not', 'no', 'nor', 'against', 'very', 'too'}
stop_words = [w for w in StopWordsRemover.loadDefaultStopWords('english') if w not in KEEP_WORDS]

tokenizer = RegexTokenizer(inputCol='text_clean', outputCol='tokens_all', pattern=r'\s+', minTokenLength=2)
remover = StopWordsRemover(inputCol='tokens_all', outputCol='tokens', stopWords=stop_words)

df = remover.transform(tokenizer.transform(df)).drop('tokens_all')
df = df.withColumn('word_count', F.size('tokens'))

df.select('text_clean', 'tokens', 'word_count').show(5, truncate=70)

## 9. Add the sentiment label from the star rating

| Stars | sentiment | label |
|---|---|---|
| 4-5 | positive | 2 |
| 3 | neutral | 1 |
| 1-2 | negative | 0 |

`label` is the number Spark MLlib needs in Phase 4.

In [ ]:
df = (df
      .withColumn('sentiment', F.when(F.col('rating') >= 4, 'positive')
                                .when(F.col('rating') == 3, 'neutral')
                                .otherwise('negative'))
      .withColumn('label', F.when(F.col('rating') >= 4, 2.0)
                            .when(F.col('rating') == 3, 1.0)
                            .otherwise(0.0)))

reviews_clean = df.select(
    'review_id', 'user_id', 'parent_asin', 'asin',
    'rating', 'sentiment', 'label',
    'title', 'text', 'text_clean', 'tokens', 'word_count',
    'verified_purchase', 'helpful_vote',
    'timestamp', 'review_date', 'year',
).cache()

log_step('6. Final clean reviews', reviews_clean)
reviews_clean.groupBy('sentiment').count().orderBy('count', ascending=False).show()

## 10. Clean the product metadata

- One row per product
- `price` text -> number (missing prices become empty / null)
- Missing category -> `Unknown`

In [ ]:
products_clean = (meta_raw
    .dropna(subset=['parent_asin'])
    .dropDuplicates(['parent_asin'])
    .select(
        'parent_asin',
        F.col('title').alias('product_title'),
        F.coalesce(F.col('main_category'), F.lit('Unknown')).alias('main_category'),
        'average_rating',
        'rating_number',
        F.col('price').cast('double').alias('price'),
        'store',
        'categories',
    )).cache()

print('Products:', f'{products_clean.count():,}')
products_clean.show(5, truncate=40)

# every review should have its product in the metadata
missing = reviews_clean.join(products_clean, 'parent_asin', 'left_anti').count()
print('Reviews whose product is missing from metadata:', missing)

## 11. Save as Parquet

- Reviews are **partitioned by year** (one folder per year), as planned in the synopsis.
  Queries like "reviews from 2020" then read only that folder.
- Written to the local disk first, then copied to Drive (Spark writing straight to Drive is slow).

In [ ]:
(reviews_clean
    .repartition('year')
    .write.mode('overwrite')
    .partitionBy('year')
    .parquet(f'{OUT_LOCAL}/reviews_clean_{SAMPLE}'))

(products_clean
    .coalesce(1)
    .write.mode('overwrite')
    .parquet(f'{OUT_LOCAL}/products_clean_{SAMPLE}'))

!mkdir -p {PROJECT}/parquet
!rm -rf {PROJECT}/parquet/reviews_clean_{SAMPLE} {PROJECT}/parquet/products_clean_{SAMPLE}
!cp -r {OUT_LOCAL}/reviews_clean_{SAMPLE} {OUT_LOCAL}/products_clean_{SAMPLE} {PROJECT}/parquet/
!du -sh {PROJECT}/parquet/*

## 12. Save the cleaning log (for the report)

In [ ]:
import pandas as pd

log_df = pd.DataFrame(cleaning_log, columns=['step', 'rows'])
log_df = log_df.drop_duplicates('step', keep='last').sort_values('step').reset_index(drop=True)
log_df['removed'] = (log_df['rows'].shift(1) - log_df['rows']).fillna(0).astype(int)

os.makedirs(f'{PROJECT}/results', exist_ok=True)
log_df.to_csv(f'{PROJECT}/results/cleaning_log_{SAMPLE}.csv', index=False)
log_df

## 13. Check: read back from Drive and query with Spark SQL

`createOrReplaceTempView` gives the tables a name so we can use SQL on them - the same
idea as Hive tables. (Real Hive tables come in Phase 5 on the WSL2 setup.)

In [ ]:
reviews = spark.read.parquet(f'{PROJECT}/parquet/reviews_clean_{SAMPLE}')
products = spark.read.parquet(f'{PROJECT}/parquet/products_clean_{SAMPLE}')
reviews.createOrReplaceTempView('reviews_clean')
products.createOrReplaceTempView('products_clean')

print('Reviews read back :', f'{reviews.count():,}')
print('Products read back:', f'{products.count():,}')

spark.sql('''
    SELECT sentiment,
           COUNT(*)                                   AS reviews,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
           ROUND(AVG(word_count), 1)                  AS avg_words,
           ROUND(100 * AVG(CAST(verified_purchase AS INT)), 1) AS verified_pct
    FROM reviews_clean
    GROUP BY sentiment
    ORDER BY reviews DESC
''').show()

In [ ]:
spark.sql('''
    SELECT year, COUNT(*) AS reviews, ROUND(AVG(rating), 2) AS avg_rating
    FROM reviews_clean
    GROUP BY year
    ORDER BY year
''').show(30)

In [ ]:
# the 10 most reviewed products in the sample, with their names
spark.sql('''
    SELECT p.product_title, p.main_category, COUNT(*) AS reviews, ROUND(AVG(r.rating), 2) AS avg_rating
    FROM reviews_clean r
    JOIN products_clean p ON r.parent_asin = p.parent_asin
    GROUP BY p.product_title, p.main_category
    ORDER BY reviews DESC
    LIMIT 10
''').show(truncate=50)

## Done

Your Drive now has:
```
amazon_reviews/
  parquet/reviews_clean_1pct/   (one folder per year)
  parquet/products_clean_1pct/
  results/cleaning_log_1pct.csv
```

**Send back:** the cleaning log table (section 12) and the outputs of section 13.
**Next:** Phase 3 - Exploratory Data Analysis (charts for the report).